# DPO's foundation: solve the finite KL optimum

Day 17 · Chapter 11. CPU, authored synthetic data, no downloads. Predict before each reference, change one declared variable, and interpret the result. Code is complete so the notebook runs from beginning to end. Completing its execution is material verification, not an assessment of learner mastery.

[Canonical chapter](../../book/chapters/11-direct-preference-optimization.md)


In [ ]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
while not (root / "src" / "dongxi_llms").is_dir():
    if root.parent == root:
        raise RuntimeError("Open this notebook within the Dongxi_LLMs repository")
    root = root.parent
sys.path.insert(0, str(root / "src"))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
                     "font.family": "DejaVu Sans", "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
blue, orange, green = "#356a9b", "#b97139", "#3d8268"



### Prediction

## Exercise 1: probability movement

A reference favors a familiar answer. Reward favors another. Predict what happens as beta approaches zero or becomes large. Then inspect the exact optimizer rather than taking a noisy training run as the answer.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
from dongxi_llms.dpo_lab import optimal_policy
reference = torch.tensor([.6,.3,.1],dtype=torch.float64)
reward = torch.tensor([0.,.5,1.],dtype=torch.float64)
betas=torch.logspace(-1,1,100,dtype=torch.float64)
policies=torch.stack([optimal_policy(reference,reward,float(b)) for b in betas])
fig,ax=plt.subplots(figsize=(7,4))
for action in range(3):ax.plot(betas,policies[:,action],label=f"answer {action}, reward {reward[action]:g}")
ax.set(xscale="log",xlabel="beta (reward/KL coefficient)",ylabel="optimal answer probability");ax.legend();plt.show()
print("Reference probabilities",reference.tolist())


### Reference explanation

**Read the figure:** larger beta moves the optimum toward the reference; smaller beta favors the highest reward. This is a fixed-reward comparison, not the initial DPO gradient's beta scaling.


![Saved reference plot for 01 kl regularized optimum, figure 1](../figures/chapter-11/day-17-01_kl_regularized_optimum-01.png)

Saved reference output from the verified CPU run. Run the preceding cell to regenerate the current experiment; this image does not change when parameters are edited.


### Prediction

## Exercise 2: prove stationarity numerically

At the optimum, reward minus beta times log-ratio-plus-one should be the same for every answer. Why is the shared constant allowed?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
beta=.5
policy=optimal_policy(reference,reward,beta)
stationarity=reward-beta*(policy.log()-reference.log()+1)
torch.testing.assert_close(stationarity,stationarity.mean().expand_as(stationarity))
print({"policy":policy.tolist(),"stationarity":stationarity.tolist(),
       "KL":float((policy*(policy.log()-reference.log())).sum()),
       "objective":float((policy*reward).sum()-beta*(policy*(policy.log()-reference.log())).sum())})


### Reference explanation

**Solution:** the probability-sum constraint supplies one Lagrange multiplier. It makes the unconstrained derivative the same across coordinates instead of forcing every coordinate derivative to zero.


### Prediction

## Exercise 3: partition cancellation and gauge

Add 50 to all rewards. Predict what changes in the policy and pair probabilities.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
offset_policy=optimal_policy(reference,reward+50,beta)
torch.testing.assert_close(offset_policy,policy)
implicit=beta*(policy.log()-reference.log())
pair_difference=implicit[2]-implicit[0]
torch.testing.assert_close(pair_difference,reward[2]-reward[0])
print({"reward_difference":float(reward[2]-reward[0]),"implicit_difference":float(pair_difference)})
try:
    optimal_policy(torch.tensor([.5,.5,0.]),torch.tensor([0.,1.,2.]),beta)
except ValueError as error:
    print("Support guard:",error)


### Reference explanation

**Solution:** the partition absorbs the offset; within-prompt reward differences survive. Full positive support is part of this finite implementation. The chart is an exact categorical result; shared neural parameters, finite labels, and incomplete optimization prevent automatic transfer of its optimum guarantee to a transformer.
